# Etapa 5a — Margen de no inferioridad para Matthews

**Pregunta.** El plan usa un margen de −0,02 para declarar que una candidata no es peor que otra. En el análisis de la partición (21/10) una diferencia de Matthews entre dos variantes **distintas** tuvo un intervalo de ± 0,033 con 18 personas, y quedó la duda de si ese margen sirve para elegir en la etapa 5. Pero en la etapa 5 las candidatas que se comparan suelen ser **parecidas** (umbrales vecinos, tamaños de limpieza), y en ese caso las diferencias imagen por imagen varían mucho menos.

**Cómo se mide.** Con desarrollo y una segmentación simple que no es candidata del trabajo (tercera componente a 1/4 de resolución, umbral de Otsu dentro del campo detectado), se arman variantes cercanas y lejanas, se calcula Matthews por imagen y la diferencia con la versión base, y se mira el ancho del intervalo al 95 % remuestreando las 18 personas. Lo mismo con el Dice de línea central. Todo a 1/4 de resolución, porque solo interesa cuánto varían las diferencias.

**Qué decide.** Un margen sirve si, para dos candidatas casi iguales, el intervalo de la diferencia es claramente más angosto que el margen; si no, la regla nunca podría declarar no inferior a nada.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage as ndi
from skimage.filters import threshold_otsu
from skimage.morphology import remove_small_objects

from src.color import color_maps
from src.data import dev_ids, load_fov, load_image, load_split, load_vessels
from src.fov import detect_fov_robust
from src.metrics import centerline_dice, matthews
from src.stats import percentile_interval, person_bootstrap_means

F = 4
ids = dev_ids()
persons = load_split().set_index("image").loc[ids, "person"]

def reduce_mask(m):
    return np.asarray(Image.fromarray(m.astype(np.uint8) * 255).reduce(F)) > 127

In [2]:
rows = []
for image_id in ids:
    rgb = np.asarray(Image.fromarray(load_image(image_id)).reduce(F))
    fov = reduce_mask(load_fov(image_id))[: rgb.shape[0], : rgb.shape[1]]
    ves = reduce_mask(load_vessels(image_id))[: rgb.shape[0], : rgb.shape[1]]
    region = detect_fov_robust(rgb)
    maps = color_maps(rgb, region)
    pc3 = maps["PC3"]
    t = threshold_otsu(pc3[region])
    green = maps["Verde"]
    g = np.where(region, green, green[region].mean())
    gc = ndi.median_filter(g, 25) - g
    variants = {
        "base": (pc3 > t) & region,
        "umbral × 1,02": (pc3 > 1.02 * t) & region if t > 0 else (pc3 > t + 0.02 * abs(t)) & region,
        "umbral × 1,05": (pc3 > t + 0.05 * abs(t)) & region,
        "umbral × 1,10": (pc3 > t + 0.10 * abs(t)) & region,
        "limpieza 10 píxeles": remove_small_objects((pc3 > t) & region, 10),
        "limpieza 40 píxeles": remove_small_objects((pc3 > t) & region, 40),
        "otro método (verde corregido)": (gc > threshold_otsu(gc[region])) & region,
    }
    for name, pred in variants.items():
        rows.append({"image": image_id, "variant": name,
                     "matthews": matthews(pred, ves, fov),
                     "centerline_dice": centerline_dice(pred, ves, fov)})
    print(image_id, end=" ", flush=True)
res = pd.DataFrame(rows)
res.to_csv(ROOT / "results" / "05a_margen.csv", index=False)

C:\Users\afrat\AppData\Local\Temp\ipykernel_24504\303548639.py:18: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  "limpieza 10 píxeles": remove_small_objects((pc3 > t) & region, 10),
C:\Users\afrat\AppData\Local\Temp\ipykernel_24504\303548639.py:19: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  "limpieza 40 píxele

01_h 

C:\Users\afrat\AppData\Local\Temp\ipykernel_24504\303548639.py:18: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  "limpieza 10 píxeles": remove_small_objects((pc3 > t) & region, 10),
C:\Users\afrat\AppData\Local\Temp\ipykernel_24504\303548639.py:19: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  "limpieza 40 píxele

02_h 

C:\Users\afrat\AppData\Local\Temp\ipykernel_24504\303548639.py:18: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  "limpieza 10 píxeles": remove_small_objects((pc3 > t) & region, 10),
C:\Users\afrat\AppData\Local\Temp\ipykernel_24504\303548639.py:19: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  "limpieza 40 píxele

03_h 

... (50 outputs left out)

C:\Users\afrat\AppData\Local\Temp\ipykernel_24504\303548639.py:18: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  "limpieza 10 píxeles": remove_small_objects((pc3 > t) & region, 10),
C:\Users\afrat\AppData\Local\Temp\ipykernel_24504\303548639.py:19: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  "limpieza 40 píxele

13_g 

C:\Users\afrat\AppData\Local\Temp\ipykernel_24504\303548639.py:18: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  "limpieza 10 píxeles": remove_small_objects((pc3 > t) & region, 10),
C:\Users\afrat\AppData\Local\Temp\ipykernel_24504\303548639.py:19: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  "limpieza 40 píxele

14_g 

In [3]:
out = []
for measure in ("matthews", "centerline_dice"):
    w = res.pivot(index="image", columns="variant", values=measure)
    diffs = w.drop(columns="base").sub(w["base"], axis=0)
    bs = person_bootstrap_means(diffs, persons.loc[diffs.index])
    lo, hi = percentile_interval(bs)
    for i, v in enumerate(diffs.columns):
        out.append({"medida": measure, "variante": v, "diferencia media": diffs[v].mean(),
                    "límite inferior": lo[i], "límite superior": hi[i],
                    "semiancho": (hi[i] - lo[i]) / 2,
                    "desvío de las diferencias": diffs[v].std()})
    print(f"{measure}: base media {w['base'].mean():.3f}")
table = pd.DataFrame(out)
table.round(4)

matthews: base media 0.259
centerline_dice: base media 0.251


,medida,variante,diferencia media,límite inferior,límite superior,semiancho,desvío de las diferencias
0,matthews,limpieza 10 píxeles,0.0133,0.0104,0.0163,0.0029,0.0072
1,matthews,limpieza 40 píxeles,0.0142,0.0105,0.0182,0.0038,0.0092
2,matthews,otro método (verde corregido),0.4262,0.4007,0.4508,0.0250,0.0695
3,matthews,"umbral × 1,02",0.0004,0.0003,0.0007,0.0002,0.0007
4,matthews,"umbral × 1,05",0.0012,0.0007,0.0018,0.0005,0.0016
5,matthews,"umbral × 1,10",0.0024,0.0014,0.0037,0.0011,0.0034
6,centerline_dice,limpieza 10 píxeles,0.0280,0.0237,0.0324,0.0044,0.0113
7,centerline_dice,limpieza 40 píxeles,0.0306,0.0257,0.0356,0.0049,0.0123
8,centerline_dice,otro método (verde corregido),0.3958,0.3723,0.4182,0.0230,0.0689
9,centerline_dice,"umbral × 1,02",0.0003,0.0001,0.0006,0.0002,0.0007


## Decisión (escrita después de correr)

**Semiancho del intervalo al 95 % de la diferencia de Matthews, con 18 personas:**

| Tipo de comparación | Ejemplos | Semiancho |
|---|---|---|
| Candidatas cercanas | umbral × 1,02 a × 1,10; limpieza de 10 o 40 píxeles | 0,0002 a 0,004 |
| Métodos distintos | otro mapa y otra forma de segmentar | 0,025 (en el análisis de la partición, 0,033) |

El Dice de línea central se comporta igual (0,0002 a 0,005 y 0,023).

- **Para candidatas cercanas,** que es lo más común en la etapa 5, −0,02 sobra: el intervalo es diez veces más angosto.
- **Para métodos distintos,** con −0,02 una candidata más simple que sea igual de buena **no se podría declarar no inferior nunca** (su límite inferior quedaría en −0,025 a −0,033), y la regla "la más simple entre las no inferiores" dejaría de funcionar: se elegiría siempre la mejor por azar, aunque sea más compleja.

**Se adopta un margen de −0,03 para Matthews** (y para el Dice de línea central, si se usa para elegir). Con eso, un método distinto pero igual de bueno puede pasar, y para candidatas cercanas no cambia nada. Para el área bajo la curva se mantiene −0,02, porque sus intervalos son mucho más angostos (± 0,006 a 0,008 en el análisis de la partición). Una pérdida de 0,03 de Matthews es chica en la práctica (por ejemplo, de 0,75 a 0,72).

Limitación: la segmentación usada es tosca (Matthews 0,26 la base); los semianchos de métodos distintos coinciden con los del análisis de la partición, hecho con un método mejor (0,67).

Se declara en `docs/plan_evaluacion.md` (desvío 14.4), antes de tener máscaras del algoritmo.
